# Python Web Service - Refresher & Practice Notebook

This notebook covers:
1. Python basics refresher
2. Virtual environments explained
3. FastAPI fundamentals
4. Pydantic models (like C# DTOs)
5. Testing your web service

---

## 1. Python Basics Refresher

Quick comparison with C# to help bridge concepts.

In [ ]:
# Variables - No type declaration needed (but you can add type hints)
name: str = "Luís"  # Type hint (optional, like C# but not enforced at runtime)
age = 30            # Without type hint
is_developer = True

print(f"Name: {name}, Age: {age}")  # f-strings = C# string interpolation $""

In [ ]:
# Collections

# List = C# List<T>
languages = ["C#", "Python", "JavaScript"]
languages.append("Blazor")  # .Add() in C#
print(f"Languages: {languages}")

# Dictionary = C# Dictionary<TKey, TValue>
person = {
    "name": "Luís",
    "role": "Developer",
    "skills": ["Blazor", "C#", "SQL"]
}
print(f"Person: {person}")
print(f"Name: {person['name']}")
print(f"Skills: {person.get('skills', [])}")

In [ ]:
# Functions

def greet(name: str, greeting: str = "Hello") -> str:
    """
    Greets a person.
    
    Args:
        name: The person's name
        greeting: Optional greeting prefix
    
    Returns:
        The greeting message
    """
    return f"{greeting}, {name}!"

# C# equivalent:
# public string Greet(string name, string greeting = "Hello") => $"{greeting}, {name}!";

print(greet("Luís"))
print(greet("World", "Olá"))

In [ ]:
# Classes

class Employee:
    """Employee class - similar to C# class."""
    
    def __init__(self, name: str, department: str):
        """Constructor - like C# constructor."""
        self.name = name
        self.department = department
    
    def get_info(self) -> str:
        """Instance method."""
        return f"{self.name} works in {self.department}"
    
    def __str__(self) -> str:
        """Like C# ToString() override."""
        return self.get_info()

# Usage
emp = Employee("Luís", "IT")
print(emp)
print(emp.get_info())

## 2. Virtual Environments Explained

Think of virtual environments like **isolated NuGet package folders** per project.

### Why?
- Project A needs `requests==2.25.0`
- Project B needs `requests==2.31.0`
- Without venv: They fight over the same installation
- With venv: Each project has its own copy

### Commands Reference

In [ ]:
# This cell shows commands - run these in your terminal, not here!

commands = """
# CREATE a virtual environment (run once per project)
python -m venv venv

# ACTIVATE (do this every time you work on the project)
# PowerShell:
.\\venv\\Scripts\\Activate.ps1

# CMD:
.\\venv\\Scripts\\activate.bat

# Linux/Mac:
source venv/bin/activate

# DEACTIVATE (when you're done)
deactivate

# INSTALL packages (while activated)
pip install fastapi uvicorn pydantic

# SAVE dependencies to file
pip freeze > requirements.txt

# INSTALL from requirements file (on new machine/server)
pip install -r requirements.txt

# SEE what's installed
pip list
"""

print(commands)

In [ ]:
# Check what's currently installed in your environment
import subprocess
result = subprocess.run(['pip', 'list'], capture_output=True, text=True)
print(result.stdout)

## 3. FastAPI Fundamentals

FastAPI is like **ASP.NET Core Minimal APIs** but for Python.

| C# / ASP.NET Core | Python / FastAPI |
|-------------------|------------------|
| `app.MapGet("/", ...)` | `@app.get("/")` |
| `app.MapPost("/", ...)` | `@app.post("/")` |
| DTOs / Records | Pydantic Models |
| `[FromBody]` | Automatic for Pydantic models |
| `[FromQuery]` | Function parameters |
| Swagger/OpenAPI | Built-in at `/docs` |

In [ ]:
# Let's install FastAPI and related packages
# Run this if you haven't installed them yet

!pip install fastapi uvicorn pydantic --quiet
print("✅ Packages installed!")

In [ ]:
# Basic FastAPI structure (this won't run a server in Jupyter, but shows the pattern)

from fastapi import FastAPI

# Create the app - like builder.Build() in .NET
app = FastAPI(
    title="My API",
    description="A sample API",
    version="1.0.0"
)

# GET endpoint - like app.MapGet("/", () => "Hello");
@app.get("/")
async def root():
    return {"message": "Hello World"}

# GET with path parameter - like app.MapGet("/items/{id}", (int id) => ...);
@app.get("/items/{item_id}")
async def get_item(item_id: int):
    return {"item_id": item_id}

# GET with query parameters
@app.get("/search")
async def search(q: str, limit: int = 10):
    return {"query": q, "limit": limit}

print("✅ FastAPI app defined!")
print("Routes defined:")
for route in app.routes:
    if hasattr(route, 'methods'):
        print(f"  {list(route.methods)} {route.path}")

## 4. Pydantic Models

Pydantic models are like **C# Records or DTOs with built-in validation**.

```csharp
// C# equivalent
public record PersonDto(
    [Required] string Name,
    int? Age = null,
    string Email = "default@email.com"
);
```

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional, Any
from datetime import datetime

# Basic model
class Person(BaseModel):
    name: str                              # Required
    age: Optional[int] = None              # Optional (nullable)
    email: str = "default@email.com"       # Default value

# Create from dict (like deserializing JSON)
data = {"name": "Luís", "age": 30}
person = Person(**data)  # ** unpacks the dict
print(f"Person: {person}")
print(f"Name: {person.name}")

# Convert to dict (like serializing to JSON)
print(f"As dict: {person.model_dump()}")
print(f"As JSON: {person.model_dump_json()}")

In [ ]:
# Model with validation and documentation

class Employee(BaseModel):
    """Employee data model with validation."""
    
    id: int = Field(description="Employee ID")
    name: str = Field(min_length=1, max_length=100, description="Employee name")
    email: str = Field(description="Employee email")
    department: Optional[str] = Field(default=None, description="Department name")
    salary: float = Field(gt=0, description="Salary (must be positive)")
    
    @field_validator('email')
    @classmethod
    def validate_email(cls, v: str) -> str:
        if '@' not in v:
            raise ValueError('Invalid email format')
        return v.lower()
    
    class Config:
        json_schema_extra = {
            "example": {
                "id": 1,
                "name": "John Doe",
                "email": "john@company.com",
                "department": "IT",
                "salary": 50000.0
            }
        }

# Valid data
emp = Employee(
    id=1,
    name="Luís",
    email="LUIS@COMPANY.COM",  # Will be lowercased by validator
    salary=45000
)
print(f"Valid employee: {emp}")
print(f"Email was lowercased: {emp.email}")

In [ ]:
# Test validation errors
from pydantic import ValidationError

try:
    # This will fail - invalid email
    bad_emp = Employee(
        id=1,
        name="Test",
        email="not-an-email",  # Missing @
        salary=50000
    )
except ValidationError as e:
    print("❌ Validation failed:")
    print(e)

print("\n" + "="*50 + "\n")

try:
    # This will fail - negative salary
    bad_emp = Employee(
        id=1,
        name="Test",
        email="test@company.com",
        salary=-100  # Must be > 0
    )
except ValidationError as e:
    print("❌ Validation failed:")
    print(e)

## 5. Generic Request/Response Models

These are the models from your `app.py` skeleton. Let's practice with them.

In [ ]:
from pydantic import BaseModel, Field
from typing import Any, Optional
from datetime import datetime

class GenericRequest(BaseModel):
    """Generic request model."""
    id: Optional[int] = Field(default=None, description="Optional identifier")
    name: Optional[str] = Field(default=None, description="Optional name")
    data: Optional[dict[str, Any]] = Field(default=None, description="Generic data payload")

class GenericResponse(BaseModel):
    """Generic response model."""
    success: bool = Field(description="Whether the operation was successful")
    message: str = Field(description="Response message")
    timestamp: datetime = Field(default_factory=datetime.now)
    request_id: Optional[int] = Field(default=None)
    result: Optional[dict[str, Any]] = Field(default=None)

# Simulate receiving a request
incoming_json = {
    "id": 123,
    "name": "ProcessData",
    "data": {
        "employee_id": 456,
        "action": "update_attendance",
        "values": [1, 2, 3]
    }
}

# Parse request
request = GenericRequest(**incoming_json)
print("📥 Received Request:")
print(f"   ID: {request.id}")
print(f"   Name: {request.name}")
print(f"   Data: {request.data}")

# Create response
response = GenericResponse(
    success=True,
    message="Data processed successfully",
    request_id=request.id,
    result={
        "processed_items": 3,
        "status": "completed"
    }
)

print("\n📤 Sending Response:")
print(response.model_dump_json(indent=2))

## 6. Testing Your Web Service

Once your server is running (`uvicorn app:app --reload`), use these cells to test it.

In [ ]:
# Install requests library for HTTP calls
!pip install requests --quiet
print("✅ requests installed!")

In [ ]:
import requests
import json

BASE_URL = "http://localhost:8000"

# Test health endpoint
def test_health():
    try:
        response = requests.get(f"{BASE_URL}/health")
        print("✅ Health Check:")
        print(json.dumps(response.json(), indent=2))
        return True
    except requests.exceptions.ConnectionError:
        print("❌ Server not running! Start it with:")
        print("   uvicorn app:app --reload --host 0.0.0.0 --port 8000")
        return False

test_health()

In [ ]:
# Test the main POST endpoint
def test_process_endpoint():
    payload = {
        "id": 1,
        "name": "TestRequest",
        "data": {
            "employee_id": 123,
            "action": "check_in",
            "location": "Office A"
        }
    }
    
    try:
        response = requests.post(
            f"{BASE_URL}/api/process",
            json=payload,
            headers={"Content-Type": "application/json"}
        )
        
        print(f"Status Code: {response.status_code}")
        print("\n📤 Request:")
        print(json.dumps(payload, indent=2))
        print("\n📥 Response:")
        print(json.dumps(response.json(), indent=2))
        
    except requests.exceptions.ConnectionError:
        print("❌ Server not running!")

test_process_endpoint()

In [ ]:
# Test with different payloads
def test_various_payloads():
    test_cases = [
        # Minimal payload
        {},
        
        # Only ID
        {"id": 42},
        
        # Full payload
        {
            "id": 100,
            "name": "FullTest",
            "data": {"key1": "value1", "key2": 123, "nested": {"a": 1}}
        }
    ]
    
    for i, payload in enumerate(test_cases, 1):
        print(f"\n{'='*50}")
        print(f"Test Case {i}:")
        print(f"Payload: {json.dumps(payload)}")
        
        try:
            response = requests.post(f"{BASE_URL}/api/process", json=payload)
            print(f"Status: {response.status_code}")
            print(f"Response: {json.dumps(response.json(), indent=2)}")
        except requests.exceptions.ConnectionError:
            print("❌ Server not running!")
            break

test_various_payloads()

## 7. Quick Reference Cheat Sheet

In [ ]:
cheat_sheet = """
╔══════════════════════════════════════════════════════════════════════╗
║                    PYTHON WEB SERVICE CHEAT SHEET                   ║
╠══════════════════════════════════════════════════════════════════════╣
║ VIRTUAL ENVIRONMENT                                                  ║
║   Create:      python -m venv venv                                   ║
║   Activate:    .\\venv\\Scripts\\Activate.ps1  (PowerShell)          ║
║   Deactivate:  deactivate                                            ║
║   Install:     pip install package_name                              ║
║   Save deps:   pip freeze > requirements.txt                         ║
║   Load deps:   pip install -r requirements.txt                       ║
╠══════════════════════════════════════════════════════════════════════╣
║ FASTAPI                                                              ║
║   Run server:  uvicorn app:app --reload --host 0.0.0.0 --port 8000   ║
║   Swagger UI:  http://localhost:8000/docs                            ║
║   ReDoc:       http://localhost:8000/redoc                           ║
╠══════════════════════════════════════════════════════════════════════╣
║ PYDANTIC (like C# DTOs)                                              ║
║   class MyModel(BaseModel):                                          ║
║       name: str                    # Required                        ║
║       age: Optional[int] = None    # Optional                        ║
║       email: str = "default"       # With default                    ║
║                                                                      ║
║   obj = MyModel(**dict_data)       # Deserialize                     ║
║   obj.model_dump()                 # To dict                         ║
║   obj.model_dump_json()            # To JSON string                  ║
╠══════════════════════════════════════════════════════════════════════╣
║ ENDPOINTS                                                            ║
║   @app.get("/path")               # GET endpoint                     ║
║   @app.post("/path")              # POST endpoint                    ║
║   @app.get("/items/{id}")         # Path parameter                   ║
║   @app.get("/search?q=term")      # Query parameter                  ║
║   async def handler(body: Model)  # Request body (auto-parsed)       ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(cheat_sheet)

## 8. C# to Python Quick Comparison

In [ ]:
comparison = """
┌────────────────────────────────┬────────────────────────────────┐
│           C#                   │           Python               │
├────────────────────────────────┼────────────────────────────────┤
│ var x = 5;                     │ x = 5                          │
│ string s = "hello";            │ s: str = "hello"               │
│ int? nullable = null;          │ nullable: Optional[int] = None │
├────────────────────────────────┼────────────────────────────────┤
│ List<string> items = new();    │ items: list[str] = []          │
│ items.Add("item");             │ items.append("item")           │
│ items.Count                    │ len(items)                     │
├────────────────────────────────┼────────────────────────────────┤
│ Dictionary<string, int> d;     │ d: dict[str, int] = {}         │
│ d["key"] = 1;                  │ d["key"] = 1                   │
│ d.TryGetValue("k", out v)      │ v = d.get("k", default)        │
├────────────────────────────────┼────────────────────────────────┤
│ $"Hello {name}"                │ f"Hello {name}"                │
│ Console.WriteLine(x);          │ print(x)                       │
├────────────────────────────────┼────────────────────────────────┤
│ public class Person { }        │ class Person:                  │
│ public Person(string n) { }    │ def __init__(self, n): ...     │
│ public string ToString()       │ def __str__(self): ...         │
├────────────────────────────────┼────────────────────────────────┤
│ async Task<T> Method()         │ async def method() -> T:       │
│ await SomeTask();              │ await some_task()              │
├────────────────────────────────┼────────────────────────────────┤
│ try { } catch (Ex e) { }       │ try: ... except Ex as e: ...   │
│ throw new Exception();         │ raise Exception()              │
├────────────────────────────────┼────────────────────────────────┤
│ items.Where(x => x > 5)        │ [x for x in items if x > 5]    │
│ items.Select(x => x * 2)       │ [x * 2 for x in items]         │
│ items.FirstOrDefault()         │ next(iter(items), None)        │
└────────────────────────────────┴────────────────────────────────┘
"""
print(comparison)

---

## ✅ You're Ready!

1. **Practice locally** with this notebook
2. **Follow the PLAN.md** step by step on Thursday
3. **Copy app.py** to the server and customize as needed
4. **Test with /docs** (Swagger UI) for easy debugging

Good luck! 🚀